# Load packages

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
import sys

# Figure 1

## Histograms

In [ ]:
path_ori = 'path/to/Original'
path_gse = 'path/to/GSE220622'
path_figurea = f"path/to/figures/1"

df_ori = pd.read_excel(f"{path_ori}/data.xlsx", index_col=0)
df_gse = pd.read_excel(f"{path_gse}/data.xlsx", index_col=0)

colors_ori = {
    'Control': 'chartreuse',
    'Case': 'red'
}

colors_gse = {
    'Control': 'forestgreen',
    'Case': 'darkred'
}

datasets = {}
datasets['Original'] = {
    'data': df_ori,
    'colors': colors_ori
}
datasets['GSE220622'] = {
    'data': df_gse,
    'colors': colors_gse
}

for ds_name, ds_dict in datasets.items():
    ds_data = ds_dict['data']
    hue_counts = ds_data['Status'].value_counts()
    hue_colors = ds_dict['colors']
    hue_replace = {x: f"{x} ({y})" for x, y in hue_counts.items()}
    hue_colors = {f"{x} ({y})": hue_colors[x] for x, y in hue_counts.items()}
    hue_order = [hue_replace[x] for x in ['Case', 'Control']]
    ds_data['Status'].replace(hue_replace, inplace=True)

    hist_bins = np.linspace(5, 115, 23)

    sns.set_theme(style='ticks')
    fig, ax = plt.subplots(figsize=(6, 3.5))
    histplot = sns.histplot(
        data=ds_data,
        bins=hist_bins,
        edgecolor='k',
        linewidth=1,
        x="Age",
        hue='Status',
        hue_order=hue_order,
        palette=hue_colors,
        ax=ax
    )
    histplot.set(xlim=(15, 80))
    plt.savefig(f"{path_figurea}/age_hist_{ds_name}.png", bbox_inches='tight', dpi=200)
    plt.savefig(f"{path_figurea}/age_hist_{ds_name}.pdf", bbox_inches='tight')
    plt.close(fig)

# Figure 2

## A

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/2"

df_data = pd.read_excel(f"{path_data}/data.xlsx", index_col=0)
df_stat = pd.read_excel(f"{path_data}/stat_bbs.xlsx", index_col=0)

df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA (Age)'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

bbs = df_stat.index.tolist()

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA (Age)': 'cyan'
}

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA (Age)'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, df_fig.shape[0] * 0.15))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA (Age)'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
sns.move_legend(ax, "lower center", bbox_to_anchor=(.4, 1), ncol=2, frameon=False)
plt.savefig(f"{path_figures}/A_barplot.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/A_barplot.pdf", bbox_inches='tight')
plt.close(fig)

colors_samples = {
    'Control': 'chartreuse',
    'Case': 'red'
}
    
n_rows = 1
n_cols = 6
fig_width = 15
fig_height = 3

sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), gridspec_kw={'wspace':0.05, 'hspace': 0.05}, layout='constrained')
for bb_id, bb in enumerate(bbs):
    row_id, col_id = divmod(bb_id, n_cols)
    
    if axs.ndim > 1:
        ax_target = axs[row_id, col_id]
    else:
        ax_target = axs[max(row_id, col_id)]
    
    sns.violinplot(
        data=df_data,
        x='Status',
        y=bb,
        palette=colors_samples,
        scale='width',
        order=['Control', 'Case'],
        saturation=0.75,
        ax=ax_target,
        legend=False,
        cut=0,
    )
    ax_target.set_ylabel(bb)
    ax_target.set_xlabel('')

fig.savefig(f"{path_figures}/A_violinplots.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_figures}/A_violinplots.pdf", bbox_inches='tight')
plt.close(fig)

## B

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/2"

df_pheno = pd.read_excel(f"{path_data}/data.xlsx", index_col=0)
df_stat = pd.read_excel(f"{path_data}/stat_age_accs.xlsx", index_col=0)
df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA (Age)'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat['ANCOVA (Age + BB)'] = -np.log10(df_stat['ancova_age_bb_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA (Age)': 'cyan',
    'ANCOVA (Age + BB)': 'olive',
}

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, df_fig.shape[0] * 0.13))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    linewidth=0.7,
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
plt.savefig(f"{path_figures}/B_barplot.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/B_barplot.pdf", bbox_inches='tight')
plt.close(fig)

ages_target = ['GrimAge', 'PCGrimAge', 'GrimAge2', 'PCPhenoAge', 'DNAmFitAge', 'HRSInCHPhenoAge']

colors_samples = {
    'Control': 'chartreuse',
    'Case': 'red'
}

sns.set_theme(style='ticks')
fig = plt.figure(
    figsize=(13, 11.5),
    layout="constrained"
)
ncols = 2
nrows = 3
subfigs = fig.subfigures(
    ncols=ncols,
    nrows=nrows,
    wspace=0.1,
    hspace=0.1,
)
for epiage_id, epiage in enumerate(ages_target):
    row_id, col_id = divmod(epiage_id, ncols)
    
    if ncols == 1 or nrows == 1:
        axs = subfigs[epiage_id].subplot_mosaic(
        [
            ['21', '22'],
        ],
        width_ratios=[3, 1.5],
        gridspec_kw={
        },
    )
    else:
        axs = subfigs[row_id, col_id].subplot_mosaic(
        [
            ['21', '22'],
        ],
        width_ratios=[3, 1.5],
        gridspec_kw={
        },
    )
    
    xy_min = df_pheno[['Age', epiage]].min().min()
    xy_max = df_pheno[['Age', epiage]].max().max()
    xy_ptp = xy_max - xy_min
    bisect = sns.lineplot(
        x=[xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp],
        y=[xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp],
        linestyle='--',
        color='black',
        linewidth=1.0,
        ax=axs['21']
    )
    regplot = sns.regplot(
        data=df_pheno.loc[df_pheno['Status'] == 'Control', :],
        x='Age',
        y=epiage,
        color=colors_samples['Control'],
        scatter=False,
        truncate=False,
        ax=axs['21']
    )
    scatter = sns.scatterplot(
        data=df_pheno,
        x='Age',
        y=epiage,
        hue='Status',
        palette=colors_samples,
        linewidth=0.5,
        alpha=0.75,
        edgecolor="k",
        s=35,
        hue_order=list(colors_samples.keys()),
        legend=True,
        ax=axs['21'],
    )
    axs['21'].set_xlim(xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp)
    axs['21'].set_ylim(xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp)
    
    sns.violinplot(
        data=df_pheno,
        x='Status',
        y=f"{epiage} acceleration",
        hue='Status',
        palette=colors_samples,
        density_norm='width',
        order=['Control', 'Case'],
        saturation=0.75,
        linewidth=1.0,
        ax=axs['22'],
        legend=False,
        cut=0,
    )
    axs['22'].set_ylabel(f"{epiage} acceleration")
    axs['22'].set_xlabel('')

fig.savefig(f"{path_figures}/B_distributions.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_figures}/B_distributions.pdf", bbox_inches='tight')
plt.close(fig)

## C

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/2"

df_data = pd.read_excel(f"{path_data}/data.xlsx", index_col=0)
df_stat = pd.read_excel(f"{path_data}/stat_metrics.xlsx", index_col=0)

df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA (Age)'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat['ANCOVA (Age + BB)'] = -np.log10(df_stat['ancova_age_bb_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

metrics = df_stat.index.tolist()

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA (Age)': 'cyan',
    'ANCOVA (Age + BB)': 'olive',
}

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, df_fig.shape[0] * 0.15))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
sns.move_legend(ax, "lower center", bbox_to_anchor=(.4, 1), ncol=2, frameon=False)
plt.savefig(f"{path_figures}/C_barplot.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/C_barplot.pdf", bbox_inches='tight')
plt.close(fig)

colors_samples = {
    'Control': 'chartreuse',
    'Case': 'red'
}
    
n_rows = 1
n_cols = 6
fig_width = 15
fig_height = 4

sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), gridspec_kw={'wspace':0.05, 'hspace': 0.05}, layout='constrained')
for epi_metric_id, epi_metric in enumerate(metrics):
    row_id, col_id = divmod(epi_metric_id, n_cols)
    
    if axs.ndim > 1:
        ax_target = axs[row_id, col_id]
    else:
        ax_target = axs[max(row_id, col_id)]

    sns.violinplot(
        data=df_data,
        x='Status',
        y=epi_metric,
        palette=colors_samples,
        scale='width',
        order=['Control', 'Case'],
        saturation=0.75,
        ax=ax_target,
        legend=False,
        cut=0,
    )
    ax_target.set_ylabel(epi_metric)
    ax_target.set_xlabel('')

fig.savefig(f"{path_figures}/C_violinplots.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_figures}/C_violinplots.pdf", bbox_inches='tight')
plt.close(fig)

## D

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/2"

df_data = pd.read_excel(f"{path_data}/data.xlsx", index_col=0)
df_stat = pd.read_excel(f"{path_data}/stat_cells.xlsx", index_col=0)

df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA (Age)'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat['ANCOVA (Age + BB)'] = -np.log10(df_stat['ancova_age_bb_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

cells = df_stat.index.tolist()

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA (Age)': 'cyan',
    'ANCOVA (Age + BB)': 'olive'
}

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, df_fig.shape[0] * 0.15))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
sns.move_legend(ax, "lower center", bbox_to_anchor=(.4, 1), ncol=2, frameon=False)
plt.savefig(f"{path_figures}/D_barplot.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/D_barplot.pdf", bbox_inches='tight')
plt.close(fig)

colors_samples = {
    'Control': 'chartreuse',
    'Case': 'red'
}
    
n_rows = 1
n_cols = 6
fig_width = 15
fig_height = 4

sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), gridspec_kw={'wspace':0.05, 'hspace': 0.05}, layout='constrained')
for cell_id, cell in enumerate(cells):
    row_id, col_id = divmod(cell_id, n_cols)
    
    if axs.ndim > 1:
        ax_target = axs[row_id, col_id]
    else:
        ax_target = axs[max(row_id, col_id)]
    
    sns.violinplot(
        data=df_data,
        x='Status',
        y=cell,
        palette=colors_samples,
        scale='width',
        order=['Control', 'Case'],
        saturation=0.75,
        ax=ax_target,
        legend=False,
        cut=0,
    )
    ax_target.set_ylabel(cell)
    ax_target.set_xlabel('')

fig.savefig(f"{path_figures}/D_violinplots.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_figures}/D_violinplots.pdf", bbox_inches='tight')
plt.close(fig)

## E

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/2"

df_stat = pd.read_excel(f"{path_data}/stat_episcores.xlsx", index_col=0)

df_stat['Features'] = df_stat.index.str.replace(' (EpiScores)', '')
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA (Age)'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat['ANCOVA (Age + BB)'] = -np.log10(df_stat['ancova_age_bb_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA (Age)': 'cyan',
    'ANCOVA (Age + BB)': 'olive',
}

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, df_fig.shape[0] * 0.07))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    linewidth=0.3,
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA (Age)', 'ANCOVA (Age + BB)'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
plt.savefig(f"{path_figures}/E_barplot.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/E_barplot.pdf", bbox_inches='tight')
plt.close(fig)

# Figure 3

## Auxiliary functions

In [ ]:
def check_for_nonnumeric(pd_series=None):
    if pd.to_numeric(pd_series, errors='coerce').isna().sum() == 0:
        return 0
    else:
        return 1


def gene_plot(d, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle):
    if genenames is not None and genenames == "deg":
        for i in d[geneid].unique():
            if (d.loc[d[geneid] == i, lfc].iloc[0] >= lfc_thr[0] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[0]) or \
                    (d.loc[d[geneid] == i, lfc].iloc[0] <= -lfc_thr[1] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[1]):
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is tuple:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is dict:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0],
                                  genenames[i], fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(genenames[i], xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)


def volcano(df="dataframe", lfc=None, pv=None, lfc_thr=(1, 1), pv_thr=(0.05, 0.05), color=("green", "grey", "red"),
            valpha=1, geneid=None, genenames=None, gfont=8, dim=(5, 5), ar=90, dotsize=1, markerdot="o",
            sign_line_v=False, sign_line_h=False, gstyle=1, axtickfontsize=9,
            axtickfontname="Arial", axlabelfontsize=9, axlabelfontname="Arial", axxlabel=None,
            axylabel=None, xlm=None, ylm=None, plotlegend=False, legendpos='best',
            figname='volcano', legendanchor=None,
            legendlabels=['Significant up', 'Not significant', 'Significant down'], theme=None, path='', ret=False, **kwargs):
    _x = r'$ \log_{2}(\mathrm{Fold Change})$'
    _y = r'$ -\log_{10}(\mathrm{p-value})$'
    color = color
    ax = kwargs.get('ax')
    if ax:
        plt.sca(ax)
    # check if dataframe contains any non-numeric character
    assert check_for_nonnumeric(df[lfc]) == 0, 'dataframe contains non-numeric values in lfc column'
    assert check_for_nonnumeric(df[pv]) == 0, 'dataframe contains non-numeric values in pv column'
    # this is important to check if color or logpv exists and drop them as if you run multiple times same command
    # it may update old instance of df
    df = df.drop(['color_add_axy', 'logpv_add_axy'], axis=1, errors='ignore')
    assert len(set(color)) == 3, 'unique color must be size of 3'
    df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'color_add_axy'] = color[0]  # upregulated
    #df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'size_add_axy'] = dotsize[0]
    df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'color_add_axy'] = color[2]  # downregulated
    #df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'size_add_axy'] = dotsize[2]
    df['color_add_axy'].fillna(color[1], inplace=True)  # intermediate
    #df['size_add_axy'].fillna(dotsize[1], inplace=True)  # intermediate
    df['logpv_add_axy'] = -(np.log10(np.array(df[pv].values.astype(float))))
    # plot
    assign_values = {col: i for i, col in enumerate(color)}
    color_result_num = [assign_values[i] for i in df['color_add_axy']]

    #assert len(set(color_result_num)) == 3, \
    #    'either significant or non-significant genes are missing; try to change lfc_thr or pv_thr to include ' \
    #    'both significant and non-significant genes'
    if theme == 'dark':
        plt.style.use('dark_background')
    #plt.subplots(figsize=dim)
    if plotlegend:
        s = plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                        s=dotsize, marker=markerdot)
        assert len(legendlabels) == 3, 'legendlabels must be size of 3'
        plt.legend(handles=s.legend_elements()[0], labels=legendlabels, loc=legendpos, bbox_to_anchor=legendanchor)
    else:
        plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                    s=dotsize, marker=markerdot)
    if sign_line_h:
        plt.axhline(y=-np.log10(pv_thr[0]), linestyle='--', color='black', linewidth=1)
    if sign_line_v:
        plt.axvline(x=lfc_thr[0], linestyle='--', color='black', linewidth=1)
        plt.axvline(x=-lfc_thr[1], linestyle='--', color='black', linewidth=1)
    gene_plot(df, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle)

    if axxlabel:
        _x = axxlabel
    if axylabel:
        _y = axylabel

    plt.xlabel(_x, fontsize=axlabelfontsize, fontname=axlabelfontname)
    plt.ylabel(_y, fontsize=axlabelfontsize, fontname=axlabelfontname)
    if xlm:
        plt.xlim(left=xlm[0], right=xlm[1])
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)

    else:
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    if ylm:
        plt.ylim(bottom=ylm[0], top=ylm[1])
        plt.yticks(np.arange(ylm[0], ylm[1], ylm[2]),  fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    else:
        plt.yticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    
    if ret:
        return plt.gca()
    else:
        plt.savefig(f"{path}/{figname}.png", bbox_inches='tight', dpi=400)
        plt.savefig(f"{path}/{figname}.pdf", bbox_inches='tight', dpi=400)
        plt.clf()
        plt.close()

## A

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/3"

df_dmps = pd.read_excel(f"{path_data}/limma.xlsx", index_col=0)
df_dmps.sort_values(["adj.P.Val"], ascending=[True], inplace=True)
df_dmps[r'$ -\log_{10}(\mathrm{p-value})$'] = -(np.log10(np.array(df_dmps["adj.P.Val"].values.astype(float))))
df_dmps[r'$ \log_{2}(\mathrm{Fold Change})$'] = df_dmps['logFC']
dmps = df_dmps.index[df_dmps["adj.P.Val"] < 0.05].values

fig, ax = plt.subplots(figsize=(7, 7), gridspec_kw={}, layout='constrained')
volc = volcano(
    df=df_dmps,
    lfc='logFC',
    pv='adj.P.Val',
    pv_thr=(0.05, 0.05),
    lfc_thr=(0.0, 0.0),
    path=f"{path_figures}",
    geneid='print',
    axtickfontsize=16,
    axlabelfontsize=16,
    gfont=16,
    gstyle=2,
    sign_line_h=True,
    sign_line_v=False,
    ar=0,
    color=('orange', 'gray', 'mediumblue'),
    dim=(7, 7), 
    ret=True,
    ax=ax,
    dotsize=4,
)
fig.savefig(f"{path_figures}/A.png", bbox_inches='tight', dpi=1200)
plt.close(fig)

## B

In [ ]:
path_data = 'path/to/Original'
path_figures = f"path/to/figures/3"

df_data = pd.read_excel(f"{path_data}/data.xlsx", index_col=0)
df_dimred = pd.read_excel(f"{path_data}/dimred.xlsx", index_col=0)
df_dimred.loc[df_dimred.index, 'Status'] = df_data.loc[df_dimred.index, 'Status']

colors_samples = {
    'Control': 'chartreuse',
    'Case': 'red'
}

dim_red_models = [
    't-SNE',
    'PCA',
    'IsoMap',
    'MDS',
]

n_rows = 2
n_cols = 2
fig_height = 8
fig_width = 8
sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), gridspec_kw={'wspace':0.05, 'hspace': 0.05}, sharey=False, sharex=False, layout='constrained')
for drm_id, drm in enumerate(dim_red_models):
    row_id, col_id = divmod(drm_id, n_cols)
    scatter = sns.scatterplot(
        data=df_dimred,
        x=f"{drm} 1",
        y=f"{drm} 2",
        hue='Status',
        palette=colors_samples,
        linewidth=0.25,
        alpha=0.75,
        edgecolor="k",
        s=40,
        ax=axs[row_id, col_id],
    )
    axs[row_id, col_id].set_title(drm)   
fig.savefig(f"{path_figures}/B.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_figures}/B.pdf", bbox_inches='tight')
plt.close(fig)

# Figure 4

## Auxiliary functions

In [ ]:
def check_for_nonnumeric(pd_series=None):
    if pd.to_numeric(pd_series, errors='coerce').isna().sum() == 0:
        return 0
    else:
        return 1


def gene_plot(d, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle):
    if genenames is not None and genenames == "deg":
        for i in d[geneid].unique():
            if (d.loc[d[geneid] == i, lfc].iloc[0] >= lfc_thr[0] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[0]) or \
                    (d.loc[d[geneid] == i, lfc].iloc[0] <= -lfc_thr[1] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[1]):
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is tuple:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is dict:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0],
                                  genenames[i], fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(genenames[i], xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)


def volcano(df="dataframe", lfc=None, pv=None, lfc_thr=(1, 1), pv_thr=(0.05, 0.05), color=("green", "grey", "red"),
            valpha=1, geneid=None, genenames=None, gfont=8, dim=(5, 5), ar=90, dotsize=1, markerdot="o",
            sign_line_v=False, sign_line_h=False, gstyle=1, axtickfontsize=9,
            axtickfontname="Arial", axlabelfontsize=9, axlabelfontname="Arial", axxlabel=None,
            axylabel=None, xlm=None, ylm=None, plotlegend=False, legendpos='best',
            figname='volcano', legendanchor=None,
            legendlabels=['Significant up', 'Not significant', 'Significant down'], theme=None, path='', ret=False, **kwargs):
    _x = r'$ \log_{2}(\mathrm{Fold Change})$'
    _y = r'$ -\log_{10}(\mathrm{p-value})$'
    color = color
    ax = kwargs.get('ax')
    if ax:
        plt.sca(ax)
    # check if dataframe contains any non-numeric character
    assert check_for_nonnumeric(df[lfc]) == 0, 'dataframe contains non-numeric values in lfc column'
    assert check_for_nonnumeric(df[pv]) == 0, 'dataframe contains non-numeric values in pv column'
    # this is important to check if color or logpv exists and drop them as if you run multiple times same command
    # it may update old instance of df
    df = df.drop(['color_add_axy', 'logpv_add_axy'], axis=1, errors='ignore')
    assert len(set(color)) == 3, 'unique color must be size of 3'
    df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'color_add_axy'] = color[0]  # upregulated
    #df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'size_add_axy'] = dotsize[0]
    df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'color_add_axy'] = color[2]  # downregulated
    #df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'size_add_axy'] = dotsize[2]
    df['color_add_axy'].fillna(color[1], inplace=True)  # intermediate
    #df['size_add_axy'].fillna(dotsize[1], inplace=True)  # intermediate
    df['logpv_add_axy'] = -(np.log10(np.array(df[pv].values.astype(float))))
    # plot
    assign_values = {col: i for i, col in enumerate(color)}
    color_result_num = [assign_values[i] for i in df['color_add_axy']]

    #assert len(set(color_result_num)) == 3, \
    #    'either significant or non-significant genes are missing; try to change lfc_thr or pv_thr to include ' \
    #    'both significant and non-significant genes'
    if theme == 'dark':
        plt.style.use('dark_background')
    # plt.subplots(figsize=dim)
    if plotlegend:
        s = plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                        s=dotsize, marker=markerdot)
        assert len(legendlabels) == 3, 'legendlabels must be size of 3'
        plt.legend(handles=s.legend_elements()[0], labels=legendlabels, loc=legendpos, bbox_to_anchor=legendanchor)
    else:
        plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                    s=dotsize, marker=markerdot)
    if sign_line_h:
        plt.axhline(y=-np.log10(pv_thr[0]), linestyle='--', color='black', linewidth=1)
    if sign_line_v:
        plt.axvline(x=lfc_thr[0], linestyle='--', color='black', linewidth=1)
        plt.axvline(x=-lfc_thr[1], linestyle='--', color='black', linewidth=1)
    gene_plot(df, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle)

    if axxlabel:
        _x = axxlabel
    if axylabel:
        _y = axylabel

    plt.xlabel(_x, fontsize=axlabelfontsize, fontname=axlabelfontname)
    plt.ylabel(_y, fontsize=axlabelfontsize, fontname=axlabelfontname)
    if xlm:
        plt.xlim(left=xlm[0], right=xlm[1])
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)

    else:
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    if ylm:
        plt.ylim(bottom=ylm[0], top=ylm[1])
        plt.yticks(np.arange(ylm[0], ylm[1], ylm[2]),  fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    else:
        plt.yticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    
    if ret:
        return plt.gca()
    else:
        plt.savefig(f"{path}/{figname}.png", bbox_inches='tight', dpi=400)
        plt.savefig(f"{path}/{figname}.pdf", bbox_inches='tight', dpi=400)
        plt.clf()
        plt.close()

## A

In [ ]:
path_data = "path/to/GSE220622"
path_figures = f"path/to/figures/4"

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA': 'cyan'
}

df_stat = pd.read_excel(f"{path_data}/stat_age_accs.xlsx", index_col=0)
df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
ages_height = df_fig.shape[0] * 0.11
fig, ax = plt.subplots(figsize=(3, ages_height))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    linewidth=0.7,
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
plt.savefig(f"{path_figures}/A.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/A.pdf", bbox_inches='tight')
plt.close(fig)

## B

In [ ]:
path_data = "path/to/GSE220622"
path_figures = f"path/to/figures/4"

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA': 'cyan'
}

df_stat = pd.read_excel(f"{path_data}/stat_metrics.xlsx", index_col=0)
df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
metrics_height = df_fig.shape[0] * 0.12
fig, ax = plt.subplots(figsize=(3, metrics_height))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
sns.move_legend(ax, "lower center", bbox_to_anchor=(.4, 1), ncol=2, frameon=False)
plt.savefig(f"{path_figures}/B.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/B.pdf", bbox_inches='tight')
plt.close(fig)

## C

In [ ]:
path_data = "path/to/GSE220622"
path_figures = f"path/to/figures/4"

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA': 'cyan'
}

df_stat = pd.read_excel(f"{path_data}/stat_cells.xlsx", index_col=0)
df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
cells_height = df_fig.shape[0] * 0.12
fig, ax = plt.subplots(figsize=(3, cells_height))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
sns.move_legend(ax, "lower center", bbox_to_anchor=(.4, 1), ncol=2, frameon=False)
plt.savefig(f"{path_figures}/C.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/C.pdf", bbox_inches='tight')
plt.close(fig)

## D

In [ ]:
path_data = "path/to/GSE220622"
path_figures = f"path/to/figures/4"

colors_tests = {
    'Mann-Whitney': 'fuchsia',
    'ANCOVA': 'cyan'
}

df_stat = pd.read_excel(f"{path_data}/stat_episcores.xlsx", index_col=0)
df_stat['Features'] = df_stat.index
df_stat['Mann-Whitney'] = -np.log10(df_stat['mw_pval_fdr_bh'].values)
df_stat['ANCOVA'] = -np.log10(df_stat['ancova_age_pval_fdr_bh'].values)
df_stat.sort_values(["Mann-Whitney"], ascending=[False], inplace=True)

df_fig = df_stat.copy()
df_fig = df_fig.melt(id_vars='Features', value_vars=['Mann-Whitney', 'ANCOVA'], var_name='Test', value_name=r"$-\log_{10}(\mathrm{p-value})$")
sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3.5, df_fig.shape[0] * 0.09))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x=r"$-\log_{10}(\mathrm{p-value})$",
    linewidth=0.3,
    edgecolor='black',
    palette=colors_tests,
    hue='Test',
    hue_order=['Mann-Whitney', 'ANCOVA'],
    ax=ax,
)
ax.set_ylabel('')
ax.axvline(-np.log10(0.05), color="red", linestyle="dotted", linewidth=2.0)
plt.savefig(f"{path_figures}/D.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_figures}/D.pdf", bbox_inches='tight')
plt.close(fig)

## E

In [ ]:
path_data = "path/to/GSE220622"
path_figures = f"path/to/figures/4"

df_stat = pd.read_excel(f"{path_data}/limma.xlsx", index_col=0)
df_stat["CpG"] = df_stat.index.values
df_stat.sort_values(["adj.P.Val"], ascending=[True], inplace=True)
df_stat['print'] = df_stat.apply(lambda row: f"{row['CpG'].split('_')[0]}", axis=1)
df_stat['log_pval'] = -np.log10(df_stat["adj.P.Val"])

fig, ax = plt.subplots(figsize=(8, 7), gridspec_kw={}, layout='constrained')
volc = volcano(
    df=df_stat,
    lfc='logFC',
    pv='adj.P.Val',
    pv_thr=(0.05, 0.05),
    lfc_thr=(0.0, 0.0),
    path=f"{path_figures}",
    geneid='print',
    axtickfontsize=14,
    axlabelfontsize=14,
    gfont=14,
    gstyle=2,
    sign_line_h=True,
    sign_line_v=False,
    ar=0,
    color=('orange', 'gray', 'mediumblue'),
    dim=(5, 6), 
    ret=True,
    ax=ax,
    dotsize=4,
)
fig.savefig(f"{path_figures}/E.png", bbox_inches='tight', dpi=1200)
plt.close(fig)